# Step 10: Depop link generation

This is where everything built so far comes together: the top item category (Step 6), the trained usage classifier (Step 8), and the garment-detail terms (Step 9) combine into an actual, clickable `depop.com/search/?q=...` URL. No model calls are needed for the URL itself -- just string construction -- but we do reuse the cached embeddings and the trained classifier one more time to gather the pieces.

## Load everything we've cached

The vibe vector, the item pool (for re-deriving the top category), and the garment-detail terms saved in notebook 07.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

vibe_vector = np.load(Path("../data/embeddings/board_vibe_vector.npy"))
item_embeddings = np.load(Path("../data/embeddings/item_embeddings.npy"))
item_ids = np.load(Path("../data/embeddings/item_ids.npy"))

with open(Path("../data/embeddings/garment_terms.json")) as f:
    garment_data = json.load(f)
    top_terms = garment_data["top_terms"]
    all_terms_ranked = garment_data["all_terms_ranked"]
    category_gate = garment_data["category_gate"]

with open(Path("../data/embeddings/aesthetic_tags.json")) as f:
    aesthetic_data = json.load(f)
    top_aesthetic = aesthetic_data["top_tag"]        # e.g. "clean girl"
    aesthetic_top_5 = aesthetic_data["top_5"]

dataset_root = Path.home() / ".cache/kagglehub/datasets/paramaggarwal/fashion-product-images-small/versions/1"
styles_df = pd.read_csv(dataset_root / "styles.csv", on_bad_lines="skip").set_index("id")

print("Top aesthetic tag:  ", top_aesthetic)
print("Top garment terms:  ", top_terms)
print("Total ranked terms: ", len(all_terms_ranked))
print("Categories with a gate:", len(category_gate))

Top aesthetic tag:   clean girl
Top garment terms:   ['cropped', 'square neck', 'off the shoulder', 'puff sleeve', 'high-waisted']
Total ranked terms:  27
Categories with a gate: 90


## Get the top recommended items

Same cosine similarity ranking as notebook 04. Unlike our first pass at this notebook, we don't collapse the top 15 matches down to one majority-vote category -- we keep each item's own individual category, so the next step can build one link per item instead of one link for the whole board. This also means the "Shorts"/"Tunics"/"Camisoles" items that a majority vote would have discarded stay represented.

In [2]:
item_norms = np.linalg.norm(item_embeddings, axis=1)
vibe_norm = np.linalg.norm(vibe_vector)
similarities = (item_embeddings @ vibe_vector) / (item_norms * vibe_norm)

TOP_N_ITEMS = 15
top_indices = np.argsort(-similarities)[:TOP_N_ITEMS]
top_recommended_ids = item_ids[top_indices]
top_scores = similarities[top_indices]

recommendations = styles_df.reindex(top_recommended_ids)[["productDisplayName", "articleType"]].copy()
recommendations["similarity"] = top_scores

recommendations

,productDisplayName,articleType,similarity
id,,,
5153,Wrangler Women's Side Paisley White T-shirt,Tshirts,0.716872
6199,UCB Women's Basic Spaghetti White Top,Tops,0.713825
57106,Elle Women White Top,Tops,0.712087
6312,UCB Women's Basic Racerback White Top,Tops,0.711823
57896,Femella Women Off White Shorts,Shorts,0.710365
44706,Wills Lifestyle Women White Top,Tops,0.709850
57180,Elle Women Blue Denim Tunic,Tunics,0.700616
57094,Elle Women Greyish Blue Culottes,Shorts,0.698559
6703,UCB Women's Epaulettes On Sleeve White T-shirt,Tshirts,0.690811


## Run the trained classifier on the vibe vector

Back in notebook 06, we saved the usage classifier specifically so it could be reused for inference without retraining. This is that moment: we rebuild the same tiny network, load its saved weights, and run the board's own vibe vector through it -- giving one more tag ("this board reads as Casual", etc.) as a bonus, alongside the search link itself. We have to apply the *same* scaling the training data used (the saved `scaler_mean`/`scaler_scale`), since the model only ever saw standardized inputs during training.

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class UsageClassifier(nn.Module):
    def __init__(self, input_dim=512, hidden_dim=128, num_classes=5):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, x):
        return self.net(x)

checkpoint = torch.load(Path("../data/embeddings/usage_classifier.pt"), weights_only=False)

classifier = UsageClassifier(
    input_dim=checkpoint["input_dim"],
    hidden_dim=checkpoint["hidden_dim"],
    num_classes=checkpoint["num_classes"],
)
classifier.load_state_dict(checkpoint["model_state_dict"])
classifier.eval()

# Apply the same standardization the training data used
vibe_scaled = (vibe_vector - checkpoint["scaler_mean"]) / checkpoint["scaler_scale"]
vibe_scaled_tensor = torch.tensor(vibe_scaled, dtype=torch.float32).unsqueeze(0)  # add batch dim

with torch.no_grad():
    logits = classifier(vibe_scaled_tensor)
    probs = F.softmax(logits, dim=1)[0]

predicted_idx = probs.argmax().item()
predicted_usage = checkpoint["class_names"][predicted_idx]

print("Predicted usage for this board:", predicted_usage)
for name, prob in zip(checkpoint["class_names"], probs.tolist()):
    print(f"  {name:<10} {prob:.3f}")

Predicted usage for this board: Sports
  Casual     0.202
  Ethnic     0.014
  Formal     0.013
  Party      0.003
  Sports     0.769


## Build the Depop search queries

Each recommended item becomes one Depop search shaped as
`{aesthetic} {garment detail} {Depop noun}` — e.g. `clean girl ribbed tee`.
Three deliberate choices behind that shape:

1. **Lead with the aesthetic.** Depop's own search is tuned for aesthetic
   vocabulary ("coquette", "y2k", "clean girl"), so the single top-ranked
   aesthetic tag from notebook 05 goes first. Physical-detail terms on their
   own ("cropped sheer") match far fewer real listings than an aesthetic does.

2. **Exactly one detail term.** Each vocabulary group (fit / neckline / fabric
   / pattern) is an attribute a garment has only *one* value of, and Depop
   search returns almost nothing for long queries. So we take just the single
   highest board-relevance term that also passes the item category's
   plausibility gate (notebook 07) — never a pile of three necklines, and
   never "square neck" on Shorts (the gate's hard subCategory rule strips
   neckline terms from anything that isn't Topwear / Innerwear / Dress).

3. **Depop vernacular for the noun.** Kaggle's `articleType` labels
   ("Tshirts", "Camisoles") aren't what sellers actually type — we map them to
   the words that appear in real listing titles ("tee", "cami").

Items that produce the same query are de-duplicated: each unique search is
shown once, with the items it covers.

In [4]:
from urllib.parse import quote_plus

DEPOP_SEARCH_BASE = "https://www.depop.com/search/"

# Kaggle articleType -> the word a Depop seller would actually put in a title.
DEPOP_NOUN = {
    "Tshirts": "tee",
    "Tops": "top",
    "Shirts": "shirt",
    "Camisoles": "cami",
    "Tunics": "tunic top",
    "Dresses": "dress",
    "Shorts": "shorts",
    "Jeans": "jeans",
    "Trousers": "trousers",
    "Track Pants": "joggers",
    "Capris": "cropped trousers",
    "Leggings": "leggings",
    "Skirts": "skirt",
    "Jackets": "jacket",
    "Blazers": "blazer",
    "Sweaters": "knit jumper",
    "Sweatshirts": "sweatshirt",
}

# Detail terms that just restate their category noun -- "cropped shorts", "knit
# sweater", "fitted leggings". The plausibility gate (nb 07) only removes terms
# that are physically impossible for a category (necklines on bottoms); it has no
# notion of an adjective being redundant with the noun. Skip these so the picker
# falls through to the next board-ranked term that actually adds information.
REDUNDANT_FOR = {
    "Shorts": {"cropped"},
    "Camisoles": {"cropped", "fitted", "spaghetti strap"},
    "Tunics": {"cropped"},
    "Leggings": {"fitted", "bodycon"},
    "Sweaters": {"knit"},
    "Jeans": {"denim"},
}

def depop_noun(article_type):
    return DEPOP_NOUN.get(article_type, article_type.lower())

def top_detail_for(article_type):
    """The single highest board-relevance garment-detail term that (a) passes
    this category's plausibility gate (nb 07) and (b) isn't redundant with the
    category noun. None if nothing qualifies."""
    gate = category_gate.get(article_type)
    passed = set(gate["passed_terms"]) if gate is not None else None
    redundant = REDUNDANT_FOR.get(article_type, set())
    for t in all_terms_ranked:            # already sorted by board relevance
        if passed is not None and t["term"] not in passed:
            continue
        if t["term"] in redundant:
            continue
        return t["term"]
    return None

def build_query_parts(article_type):
    parts = [top_aesthetic]                      # e.g. "clean girl"
    detail = top_detail_for(article_type)
    if detail:
        parts.append(detail)                    # e.g. "high-waisted"
    parts.append(depop_noun(article_type))      # e.g. "shorts"
    return parts

def build_depop_url(parts):
    return DEPOP_SEARCH_BASE + "?q=" + quote_plus(" ".join(parts))

recommendations["query_parts"] = recommendations["articleType"].apply(build_query_parts)
recommendations["query_text"] = recommendations["query_parts"].apply(" ".join)
recommendations["depop_url"] = recommendations["query_parts"].apply(build_depop_url)

recommendations[["productDisplayName", "articleType", "query_text", "depop_url"]]

,productDisplayName,articleType,query_text,depop_url
id,,,,
5153,Wrangler Women's Side Paisley White T-shirt,Tshirts,clean girl cropped tee,https://www.depop.com/search/?q=clean+girl+cro...
6199,UCB Women's Basic Spaghetti White Top,Tops,clean girl cropped top,https://www.depop.com/search/?q=clean+girl+cro...
57106,Elle Women White Top,Tops,clean girl cropped top,https://www.depop.com/search/?q=clean+girl+cro...
6312,UCB Women's Basic Racerback White Top,Tops,clean girl cropped top,https://www.depop.com/search/?q=clean+girl+cro...
57896,Femella Women Off White Shorts,Shorts,clean girl high-waisted shorts,https://www.depop.com/search/?q=clean+girl+hig...
44706,Wills Lifestyle Women White Top,Tops,clean girl cropped top,https://www.depop.com/search/?q=clean+girl+cro...
57180,Elle Women Blue Denim Tunic,Tunics,clean girl square neck tunic top,https://www.depop.com/search/?q=clean+girl+squ...
57094,Elle Women Greyish Blue Culottes,Shorts,clean girl high-waisted shorts,https://www.depop.com/search/?q=clean+girl+hig...
6703,UCB Women's Epaulettes On Sleeve White T-shirt,Tshirts,clean girl cropped tee,https://www.depop.com/search/?q=clean+girl+cro...


## Full pipeline summary

Everything the project has produced for this board, end to end.

In [5]:
unique = recommendations.groupby("query_text")
covered = unique["productDisplayName"].apply(list)
url_for = unique["depop_url"].first()

print("=" * 64)
print("BOARD VIBE SUMMARY")
print("=" * 64)
print(f"Top aesthetic:        {top_aesthetic}   (top 5: {', '.join(aesthetic_top_5)})")
print(f"Predicted usage:      {predicted_usage}")
print(f"Board garment terms:  {', '.join(top_terms)}")
print()
print(f"{len(covered)} unique Depop searches for {len(recommendations)} recommended items:\n")
for query_text in covered.index:
    print(f'  "{query_text}"')
    print(f"    {url_for[query_text]}")
    print(f"    covers: {', '.join(covered[query_text])}\n")

BOARD VIBE SUMMARY
Top aesthetic:        clean girl   (top 5: clean girl, minimalist, preppy, coquette, glam)
Predicted usage:      Sports
Board garment terms:  cropped, square neck, off the shoulder, puff sleeve, high-waisted

5 unique Depop searches for 15 recommended items:

  "clean girl cropped tee"
    https://www.depop.com/search/?q=clean+girl+cropped+tee
    covers: Wrangler Women's Side Paisley White T-shirt, UCB Women's Epaulettes On Sleeve White T-shirt, UCB Women's Basic Round Neck White T-shirt, Wrangler Women's Pinstrupe Grey Melange T-shirt, UCB Women's Basic Round Light Pink T-shirt

  "clean girl cropped top"
    https://www.depop.com/search/?q=clean+girl+cropped+top
    covers: UCB Women's Basic Spaghetti White Top, Elle Women White Top, UCB Women's Basic Racerback White Top, Wills Lifestyle Women White Top, Femella Women Grey Top, Forever New Women Ditsy Print White Top

  "clean girl high-waisted shorts"
    https://www.depop.com/search/?q=clean+girl+high-waisted+sh